# A Modern Solution to a Modern Problem

## Find jobs quick and enter the market quicker.

### CHALLENGE:

It is hard to keep track of the job market and find the right jobs to apply for.

You can now find the right jobs that match your resume in few minutes across domains and eliminate duplicate applications. 

Simplify the process and get more done!

In [34]:
import os
import json
import asyncio
from typing import Dict
import sendgrid
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from sendgrid.helpers.mail import Mail, Email, To, Content
from scraper import fetch_website_links, fetch_website_contents
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from openai import OpenAI
from pypdf import PdfReader

In [35]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4.1'
openai = OpenAI()

API key looks good so far


## First step: Fetch the domain and gather all the job recent job applications in your domain. 

### Use the Beautiful Soup library to scrape your favorite job board and gather all the recent posts. 

It will gather all the links posted in your domain that is aimed for juniors and associates posted within one day of posting.

In [36]:
link_system_prompt = """
You are provided with a list of links found on a job board webpage.
You are able to decide which of the links would be most relevant to a job posting,
such as links to specific job postings, or links to the company's website.
Ensure that the job description is a summary of the job duties and not the entire job description.
You should respond in JSON as in this example:

{
    "links": [
        {"Company name": "name", "Job title": "title", "Job duties": "summary of the job duties", "url": "https://full.url/goes/here/about"},
        {"Company name 2": "name 2", "Job title 2": "title", "Job duties 2": "summary of the job duties 2", "url": "https://another.full.url/careers"}
    ]
}

- Always respond in JSON format.
- Always include company name, job title, job duties, and url.
"""

In [37]:
@function_tool
def get_links_user_prompt(url: str) -> str:
    """Relevant links finder"""
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a job posting, not the company's website.
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links, or links to the company's website.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [38]:
get_links_user_prompt
#"https://www.linkedin.com/jobs/search/?f_E=2%2C3&f_TPR=r86400&geoId=104769905&keywords=""ai""&origin=JOB_SEARCH_PAGE_JOB_FILTER&refresh=true"

FunctionTool(name='get_links_user_prompt', description='Relevant links finder', params_json_schema={'properties': {'url': {'title': 'Url', 'type': 'string'}}, 'required': ['url'], 'title': 'get_links_user_prompt_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x126346f90>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [39]:
@function_tool
def select_relevant_links(url: str) -> dict:
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links


In [40]:
select_relevant_links
#"https://www.linkedin.com/jobs/search/?f_E=2%2C3&f_TPR=r86400&geoId=104769905&keywords=""ai""&origin=JOB_SEARCH_PAGE_JOB_FILTER&refresh=true"

FunctionTool(name='select_relevant_links', description='', params_json_schema={'properties': {'url': {'title': 'Url', 'type': 'string'}}, 'required': ['url'], 'title': 'select_relevant_links_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x126346740>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

## Second step: match with personal experience!

# Filter the jobs to match your experience!

Update your resume.pdf and the LLM will make the decision for you and reason why you should apply. 

In [41]:
name = "Sriram Karthikeyan"

In [42]:
reader = PdfReader("Personal_details/resume.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

In [43]:
print(resume)

Sriram Karthikeyan
sriramkofc@gmail.com
 
+61 426354114
 
Sydney, NSW
 
https://www.linkedin.com/in/sriramkarthikeyan/
 
PROFILE
Working on production AI systems used within financial advisory workflows, with a focus on reliability, 
compliance, and practical business value. Designed and implemented RAG pipelines to analyse advice 
transcripts and supporting documents, enabling context-aware reasoning and structured insight extraction. 
Built agent-style automation logic to assess intent, tone, and risk indicators, supporting internal compliance 
and quality review processes. Developed end-toend automated workflows covering file ingestion, analysis, 
validation, and reporting, reducing manual effort and improving turnaround times. Collaborate closely with 
product, compliance, and leadership stakeholders to ensure AI outputs align with regulatory and 
operational requirements.
EDUCATION
08/2022 – 07/2024
Sydney, NSW
Masters in IT and IT Management, The University of Sydney
Data Managem

In [44]:
create_job_list_instructions = f"""
You create a shortlist of jobs for {name}.

Workflow:
1. Use the link tools to gather and filter relevant links.
2. Use suggestion prompt tool to construct the recommendation prompt.
3. Return a concise shortlist with company, role, reason, and url.
"""

create_job_list_agent = Agent(
    name="Job List Creator",
    instructions=create_job_list_instructions,
    tools=[select_relevant_links, get_links_user_prompt],
    model="gpt-4.1"
)

@function_tool
async def create_job_list(url: str) -> str:
    """Find and shortlist relevant jobs from the exact input URL."""
    result = await Runner.run(
        create_job_list_agent,
        f"Use this exact URL as the source and do not switch to another one: {url}"
    )
    return result.final_output if hasattr(result, "final_output") else str(result)

In [45]:
create_job_list

FunctionTool(name='create_job_list', description='Find and shortlist relevant jobs from the exact input URL.', params_json_schema={'properties': {'url': {'title': 'Url', 'type': 'string'}}, 'required': ['url'], 'title': 'create_job_list_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x12573eeb0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [46]:
subject_instructions = "You can write a subject for a list of jobs. \
You are given a list and you need to write a subject for an email that states how many jobs applying for."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

job_filter_instructions = f"You are looking at a list of jobs.\
Use this information to build a shorter list of jobs that are most relevant to {name}'s career, background, skills and experience.\n\n\
    "

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for an email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

job_filter = Agent(name = "Job Filter tool", instructions = job_filter_instructions, model="gpt-4o-mini")
job_filter_tool = job_filter.as_tool(tool_name = "job_filter", tool_description="Filter out relevant jobs from a list of jobs")

In [47]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("sriramkofc@gmail.com")  # Change to your verified sender
    to_email = To("sriramkofc@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [48]:
email_tools = [subject_tool, html_tool, send_html_email,job_filter_tool]
job_hunter_tools = [create_job_list]

print(email_tools)
print(job_hunter_tools)

[FunctionTool(name='subject_writer', description='Write a subject for an email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x1255fb5b0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_i

In [49]:
instructions = "You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it"
)

In [50]:
handoffs = [emailer_agent]
print(job_hunter_tools)
print(handoffs)

[FunctionTool(name='create_job_list', description='Find and shortlist relevant jobs from the exact input URL.', params_json_schema={'properties': {'url': {'title': 'Url', 'type': 'string'}}, 'required': ['url'], 'title': 'create_job_list_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x12573eeb0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)]
[Agent(name='Email Manager', handoff_description='Convert an email to HTML and send it', tools=[FunctionTool(name='subject_writer', description='Write a subject for an email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'obj

In [ ]:
job_hunter_instructions = f"""
You are a job hunter for {name}. Your goal is to find the best jobs that {name} should apply for.

You act as a career counselor, reviewing job postings and recommending the most relevant opportunities based on {name}'s resume, background, skills, and experience.

Go through the provided job postings and select only the ones worth applying for. For each recommended job, explain your reasoning clearly.

If you are unsure whether a job is a good fit, include it with a note in the "Reason" field explaining your uncertainty. Only return jobs worth considering — omit poor matches entirely.

Follow these steps carefully:
1. Extract the URL from the user request and pass that exact URL into the create_job_list tool.
2. Prepare one clean email-ready draft of shortlisted roles.
3. Handoff ONLY that final draft to the Email Manager for formatting and sending.

Crucial Rules:
- Use tools to gather jobs; do not invent links.
- When a URL is provided by the user, use that exact URL only.
- Handoff exactly ONE final draft to Email Manager.
"""

job_hunter_instructions += f"\n\n## Resume:\n{resume}\n\n"
job_hunter_instructions += f"Use this context to make decisions on the jobs {name} should apply to. Always include company name, job title, job duties, reason and url."

job_hunter = Agent(
    name="Job Hunter",
    instructions=job_hunter_instructions,
    tools=job_hunter_tools,
    handoffs=handoffs,
    model="gpt-4o-mini"
)

Linkedin_jobs = "https://www.linkedin.com/jobs/search/?f_E=2%2C3&f_TPR=r86400&geoId=104769905&keywords=ai&origin=JOB_SEARCH_PAGE_JOB_FILTER&refresh=true"
seek_jobs = "https://www.seek.com.au/ai-jobs/in-Sydney-NSW-2000?daterange=1"

message = f"Find suitable jobs from these exact URLs: {Linkedin_jobs}, and {seek_jobs} and send me a shortlist email"

with trace("Job Postings"):
    result = await Runner.run(job_hunter, message)

display(Markdown(result.final_output if hasattr(result, "final_output") else str(result)))

Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


The email with the job shortlist has been successfully sent! If you need anything else or further assistance, just let me know!